# Step 3 · Planarian and human model results

Each step presents the planarian analysis first, followed by the separate human analysis.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
ROOT=next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p/'dataset').is_dir() and (p/'results').is_dir()), None)
assert ROOT is not None, 'Start Jupyter within the repository'
assert (ROOT/'dataset').is_dir() and (ROOT/'results').is_dir(), 'Start Jupyter from the repository root or notebooks/'
sys.path.insert(0,str(ROOT))
PROJECT=ROOT/'dataset'
PANDO=ROOT/'results/pando'
MODEL=ROOT/'results/model'
SCENIC=ROOT/'results/scenic'
SCENIC_DATA=ROOT/'dataset/scenic'
pd.set_option('display.max_columns',30)

print('Package:',ROOT)
print('Python:',sys.version.split()[0])
print('Directories available:', all(p.exists() for p in [PROJECT,PANDO,MODEL,SCENIC]))

## Planarian · predicted targets

The saved planarian edge model is applied to `SMED30006440` (annotated Hes1), ranking candidates across the saved planarian gene embedding. Scores are predicted class × class probability; they are not calibrated probabilities or proof of regulatory direction. If the model's highest class is zero, the strongest nonzero class is used as a fallback, indicated in the table. This is a selected regulator example, not a whole-network result.


In [ ]:
from matplotlib import pyplot as plt
planarian_top=pd.read_csv(ROOT/'results/planarian/phase3_output/Hes1_top10.tsv',sep='\t')
print(planarian_top[['rank','source','target','target_symbol','predicted_class','signed_score','zero_class_fallback']].to_string(index=False))
fig,ax=plt.subplots(figsize=(8,4.8)); plot=planarian_top.iloc[::-1]
labels=[f"{symbol} ({gene})" if symbol!=gene else gene for symbol,gene in zip(plot.target_symbol,plot.target)]
ax.barh(labels,plot.signed_score,color=['#c97064' if x<0 else '#5687b0' for x in plot.signed_score])
for label,value in zip(labels,plot.signed_score):
 ax.annotate(f'{value:.3f}',(value,label),xytext=(5 if value>0 else -5,0),textcoords='offset points',va='center',ha='left' if value>0 else 'right',fontsize=8)
ax.margins(x=.12)
ax.set(xlabel='Signed model score (uncalibrated)',title='Planarian Hes1-associated candidate targets')
fig.tight_layout(); out=ROOT/'results/generated';out.mkdir(exist_ok=True);fig.savefig(out/'planarian_Hes1_top10.png',dpi=180);fig.savefig(out/'planarian_Hes1_top10.pdf');plt.show()


## Human · GLI3 target rankings

We inspect the saved GLI3 ranking and the model's selected two-layer tree. The scores are uncalibrated ranking values, not probabilities of true regulation. The 1,794-gene universe includes HES4 added after its importance was known; this is targeted exploration.

In [ ]:
top_model=pd.read_csv(MODEL/'GLI3_all_targets_top10.tsv',sep='\t')
print(top_model.to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,4.8))
plot=top_model.iloc[::-1]; ax.barh(plot.target,plot.score,color=np.where(plot.hit,'#087e76','#a8b7bf'))
ax.set(xlabel='Model ranking score (uncalibrated)',title='pando+GRN: GLI3 top-ranked candidates',xlim=(0,1));fig.tight_layout()
out=ROOT/'results/generated';out.mkdir(exist_ok=True);fig.savefig(out/'model_top10.png',dpi=180);plt.show()

Candidate-table membership is highlighted. HES4 and PAX6 already occur in the first-stage Pando GLI3 edge set; neither the ranking nor the plot establishes an independent direct interaction.

In [ ]:
from matplotlib import pyplot as plt
folder=MODEL/'phase4_trees'; edges=pd.read_csv(folder/'grnm_edges.csv')
print('Displayed first-layer targets:',int(edges.layer.eq(1).sum()),'second-layer candidates:',int(edges.layer.eq(2).sum()))
fig,ax=plt.subplots(figsize=(7,7));ax.imshow(plt.imread(folder/'grnm_tree.png'));ax.axis('off');plt.show()

The tree is a selected two-layer display, not a complete or experimentally confirmed network. To rebuild it, see `src/visualization/`; a suitable R installation is required to re-render the saved edge tables.